# Loan Default Prediction — Standard ML Pipeline

**Prepared by the data team** (working with an AI coding assistant).

**For:** the go/no-go decision meeting.

**Recommendation:** deploy the neural network (accuracy ~98%).

---
Runs as-is in **Google Colab**: `Runtime → Run all`. 
Outputs: `loans.csv`, an income histogram, and a model-comparison chart (both shown inline).

In [ ]:
# Setup
# In Colab, torch / keras / plotly / sklearn / pandas are preinstalled.
# If running locally, first:  pip install torch keras plotly scikit-learn pandas
import os
os.environ["KERAS_BACKEND"] = "torch"

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import keras
from keras import layers

rng = np.random.default_rng(7)
keras.utils.set_random_seed(7)

## 1. Load data
Monthly snapshots of the bank's loans, exported from the warehouse. 
*(The warehouse stores approved loans only.)*

In [ ]:
def load_data(n_customers=1200):
    rows = []
    for cid in range(n_customers):
        self_emp = rng.random() < 0.25
        income_m = np.exp(rng.normal(9.1, 0.45))                 # monthly income
        income = income_m * 12 if rng.random() < 0.3 else income_m  # some systems store it annually
        score = np.clip(rng.normal(680, 70), 300, 850)
        loan = np.clip(rng.normal(70000, 30000), 8000, 300000)

        risk = -3.1 - 0.011 * (score - 680) + 1.6 * loan / (income_m * 12) \
               - 0.5 * (not self_emp) + rng.normal(0, 0.4)
        default = int(rng.random() < 1 / (1 + np.exp(-risk)))

        days_late = rng.uniform(2, 35) if default else rng.exponential(2.5)
        collections = int((default and rng.random() < 0.65) or rng.random() < 0.05)

        income = np.nan if rng.random() < (0.30 if self_emp else 0.05) else round(income)
        score = np.nan if rng.random() < 0.08 else round(score)

        for month in range(1, int(rng.integers(3, 9)) + 1):
            rows.append({
                "customer_id": cid,
                "month": month,
                "income": income,
                "self_employed": int(self_emp),
                "credit_score": score,
                "loan_amount": round(loan),
                "avg_days_late": round(max(0, days_late + rng.normal(0, 2)), 1),
                "collections_flag": collections,
                "default": default,
            })
    return pd.DataFrame(rows)


df = load_data()
df.to_csv("loans.csv", index=False)
print("Data:", df.shape, "| default rate: {:.1%}".format(df["default"].mean()))
df.head()

In [ ]:
px.histogram(df, x="income", nbins=80, title="Income distribution").show()

## 2. Clean

In [ ]:
df["income"] = df["income"].fillna(df["income"].mean())
df["credit_score"] = df["credit_score"].fillna(0)

X = df.drop(columns=["default", "customer_id"])
y = df["default"]
X = StandardScaler().fit_transform(X)

## 3. Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("train:", len(y_train), "| test:", len(y_test))

## 4. Model 1: Random Forest
*(model parameters: `n_estimators=100`, default settings are fine)*

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
acc_rf = accuracy_score(y_test, rf.predict(X_test))
print("Random Forest accuracy: {:.4f}".format(acc_rf))

## 5. Model 2: Neural Network
*(model parameters: 3 layers, 30 epochs, no tuning needed)*

In [ ]:
nn = keras.Sequential([
    layers.Input(shape=(X.shape[1],)),
    layers.Dense(256, activation="relu"),
    layers.Dense(256, activation="relu"),
    layers.Dense(256, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
nn.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
nn.fit(X_train, y_train, epochs=30, batch_size=64, verbose=0)
acc_nn = accuracy_score(y_test, (nn.predict(X_test, verbose=0) > 0.5).astype(int))
print("Neural Network accuracy: {:.4f}".format(acc_nn))

## 6. Compare and decide

In [ ]:
print("Random Forest  accuracy: {:.4f}".format(acc_rf))
print("Neural Network accuracy: {:.4f}".format(acc_nn))

fig = go.Figure(go.Bar(x=["Random Forest", "Neural Network"], y=[acc_rf, acc_nn]))
fig.update_yaxes(range=[0.90, 1.00])
fig.update_layout(title="Model comparison — test accuracy")
fig.show()

In [ ]:
print("Both models are around 98% - far better than a 50/50 coin flip.")
print("Decision: deploy the NEURAL NETWORK (deep learning is the more advanced technology).")